In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

LangChain의 핵심 개념인 LCEL입니다. create_agent() 안에 감춰진 원리들을 이해하기 위해선 한 번 살펴볼 필요가 있습니다.

```
chain = prompt | model | output_parser
chain.invoke({"topic": "RAG"})
```

`ChatPromptTemplate`은 재사용 가능한 프롬프트 사용을 위해 사용됩니다.

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 번역가야. 한국어를 영어로 번역해."),
    ("human", "{text}")
])
input_text = prompt.invoke({"text": "안녕하세요"})
print(input_text)

messages=[SystemMessage(content='너는 번역가야. 한국어를 영어로 번역해.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={})]


topic에 어떤 단어가 들어가냐에 따라서 프롬프트 템플릿이 완성되게 됩니다. 이런 식으로 다양한 입력에 대해서 같은 프롬프트 형식을 재사용하기 위해 사용되는 것입니다.

model은 입력 텍스트를 보고 답변을 생성하는 llm입니다.

In [ ]:
model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

response = model.invoke(input_text)
print(response)

content='Hello.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 28, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DejqYhdvjSIs40XX0wQtjUOFYLESP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1cec-92d5-7d82-b038-e1806ba84ed5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 5, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


llm의 응답은 기본적으로 객체(`AIMessage`)로 옵니다. `StrOutputParser`는 여기서 순수 텍스트 문자열만 꺼내는 역할을 합니다.

In [4]:
output_parser = StrOutputParser()

final_output = output_parser.invoke(response)
print(final_output)

Hello.


chain은 이 모든 과정을 하나의 흐름으로 묶어서 한 번에 실행할 수 있는 에이전트의 흐름을 만드는 기본 단위입니다.

prompt -> model -> output_parser

3단계의 에이전트 흐름을 하나로 묶어주는 것이죠.

In [5]:
chain = prompt | model | output_parser
response = chain.invoke({"text": "오늘은 잘 시간도 없이 바쁘네요."})
print(response)

I’m so busy today that I don’t even have time to sleep.


LCEL 체인은 단순한 순차 흐름에 적합하지만, **조건 분기·루프·상태 관리**가 필요한 복잡한 에이전트를 만들려면 **LangGraph**를 사용합니다.

LangGraph는 각 처리 단계를 **노드(node)**, 단계 사이의 연결을 **엣지(edge)**로 표현하는 그래프 구조입니다.

```
START → translate_node → END
```

아래 예시는 앞서 만든 번역 체인과 동일한 동작을 LangGraph로 구현한 것입니다.

In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 그래프가 단계 사이에서 주고받는 상태(state)를 정의합니다.
class TranslationState(TypedDict):
    text: str        # 입력 텍스트
    result: str      # 번역 결과

# 노드: 상태를 받아서 변환된 상태를 반환하는 함수입니다.
def translate_node(state: TranslationState) -> TranslationState:
    chain = prompt | model | output_parser
    translated = chain.invoke({"text": state["text"]})
    return {"result": translated}

# 그래프 정의
builder = StateGraph(TranslationState)
builder.add_node("translate", translate_node)
builder.add_edge(START, "translate")
builder.add_edge("translate", END)
# builder.add_conditional_edges()를 통해 분기 추가 가능

graph = builder.compile()

# 실행
output = graph.invoke({"text": "오늘은 잘 시간도 없이 바쁘네요."})
print(output["result"])

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


I’m so busy today that I don’t even have time to sleep.


서비스가 복잡해질수록 체인을 구성하는 것이 어려워지겠죠? create_agent()는 이 복잡한 조립을 대신 해주는 기능이라고 생각하면 됩니다. 

In [7]:
# create_agent()
from langchain.agents import create_agent

# 이 안에서 prompt | model 등의 흐름이 자동으로 정의됨.
agent = create_agent(
    model,
    system_prompt="너는 번역가야. 한국어를 영어로 번역해."
)

messages = [
    {"role": "user", "content": "오늘은 잘 시간도 없이 바쁘네요."}
]

response = agent.invoke(
    {"messages": messages}
)

print(response['messages'][-1].content)

I'm so busy today that I don't even have time to sleep.
